In [ ]:
# Week 7: Product–Warehouse Alignment & Performance Scoring

import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 1. Load Data
data_path = '../data/'
engineered_sales_df = pd.read_csv(os.path.join(data_path, 'engineered_sales.csv'))
branches_df = pd.read_csv(os.path.join(data_path, 'branches.csv'))
products_df = pd.read_csv(os.path.join(data_path, 'validated_products.csv'))

# 2. Aggregate Product-Warehouse Pair Performance
pw_matrix = engineered_sales_df.groupby(['branch_id', 'product_id']).agg(
    total_units_sold=('quantity', 'sum'),
    total_revenue=('total_revenue', 'sum'),
    order_count=('order_id', 'nunique')
).reset_index()

# 3. Compute Composite Performance Scores (0 - 100 Scale)
# Product Performance Scoring
prod_agg = pw_matrix.groupby('product_id').agg(
    prod_revenue=('total_revenue', 'sum'),
    prod_units=('total_units_sold', 'sum')
).reset_index()
prod_agg['product_score'] = (
    (prod_agg['prod_revenue'] / prod_agg['prod_revenue'].max() * 60) +
    (prod_agg['prod_units'] / prod_agg['prod_units'].max() * 40)
)

# Warehouse Performance Scoring
wh_agg = pw_matrix.groupby('branch_id').agg(
    wh_revenue=('total_revenue', 'sum'),
    wh_units=('total_units_sold', 'sum')
).reset_index()
wh_agg['warehouse_score'] = (
    (wh_agg['wh_revenue'] / wh_agg['wh_revenue'].max() * 60) +
    (wh_agg['wh_units'] / wh_agg['wh_units'].max() * 40)
)

# 4. Merge Alignment Scores into Final Matrix
alignment_summary = pw_matrix.merge(prod_agg[['product_id', 'product_score']], on='product_id', how='left')
alignment_summary = alignment_summary.merge(wh_agg[['branch_id', 'warehouse_score']], on='branch_id', how='left')

# 5. Save Processed Deliverable
output_path = '../data/processed/'
os.makedirs(output_path, exist_ok=True)
alignment_summary.to_csv(os.path.join(output_path, 'product_warehouse_alignment_summary.csv'), index=False)
print('Week 7 alignment & performance scoring pipeline completed successfully.')